## Ingestion


In [ ]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [ ]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

## Chunking and Preprocessing

### Intelligent Chunking with LLM

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# now you can access them
openai_key = os.getenv("OPENAI_API_KEY")
print("Has key?", bool(openai_key))

openai_client = OpenAI(openai_key)


def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text


In [ ]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [ ]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [ ]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
evidently_docs = read_repo_data(repo_owner, repo_name)

In [ ]:
from tqdm.auto import tqdm

evidently_chunks = []

for doc in tqdm(evidently_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

### Simple Chunking

In [ ]:
def sliding_window(seq, size, step):
  if size <= 0 or step <= 0:
    raise ValueError("size and step must be positive")

  n = len(seq)
  result = []

  for i in range(0, n, step):
    chunk = seq[i:i+size]
    result.append({'start': i, 'chunk': chunk})

    if i + size >= n:
      break
  
  return result

In [ ]:
from tqdm.auto import tqdm

evidently_chunks = []
for doc in evidently_docs:
  doc_copy = doc.copy()
  doc_content = doc_copy.pop('content')
  chunks = sliding_window(doc_content, 2000, 1000)

  for chunk in chunks:
    chunk.update(doc_copy)

  evidently_chunks.extend(chunks)